# 01: Non-Speech Data Collection

Collect non-speech audio for Vietnamese Bag-of-Hallucinations construction.

This notebook supports two execution modes:

- **Colab mode**: writes artifacts to Google Drive at `/content/drive/MyDrive/shrike-7/datasets/noise_for_boh/`.
- **Local mode**: writes artifacts to the ignored repo folder `data/noise_for_boh/`.

The local mode is useful when Colab GPU quota is unavailable. It is slower for later ASR inference, but data collection itself does not require a GPU.


## 0. Check Runtime

This cell is intentionally safe on both Colab and local machines. A missing GPU is fine for this notebook.


In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No NVIDIA GPU detected. This is expected on local Mac runs.")


## 1. Configuration

`TARGET_TOTAL_SAMPLES` caps the full collection size. ESC-50 and synthetic sources are generated in order until the target is reached.

Optional sources such as MUSAN/FSD50K are kept as disabled placeholders because their exact Hugging Face layout can change; enable them only after verifying the dataset card and split names.

In [ ]:
PROJECT = "shrike-7"
RUN_NAME = "d2_5_noise_collection"
SEED = 42
SAMPLE_RATE = 16000

# "auto" chooses Colab when google.colab is available, otherwise local.
# Set to "local" explicitly when running from VS Code/Jupyter on Mac.
EXECUTION_MODE = "auto"  # "auto" | "colab" | "local"
USE_DRIVE_STORAGE = True

# Keep this conservative for local Mac and Colab Free. Increase after a smoke run passes.
TARGET_TOTAL_SAMPLES = 800

NOISE_SOURCES = {
    "esc50": {
        "enabled": True,
        "hf_repo": "ashraq/esc50",
        "split": "train",
        "n_samples": 500,
        "exclude_categories": [
            "crying_baby",
            "sneezing",
            "clapping",
            "breathing",
            "coughing",
            "footsteps",
            "laughing",
            "brushing_teeth",
            "snoring",
            "drinking_sipping",
            "crying",
            "speaking",
        ],
    },
    "musan_noise": {
        "enabled": False,
        "hf_repo": None,
        "split": None,
        "n_samples": 500,
        "note": "Disabled until the exact HF dataset layout is verified.",
    },
    "fsd50k_subset": {
        "enabled": False,
        "hf_repo": None,
        "split": None,
        "n_samples": 0,
        "note": "Heavyweight optional source; enable only for larger BoH runs.",
    },
}

SYNTHETIC_NOISE = {
    "enabled": True,
    "n_silence": 100,
    "n_white_noise": 100,
    "n_pink_noise": 100,
    "durations_s": [1.0, 3.0, 5.0, 10.0, 20.0],
    "amplitudes": [0.001, 0.003, 0.005, 0.01],
}


## 2. Install Dependencies

The dependency cell works in both Colab and local Jupyter kernels. If you already use the repo's `.venv`, this only fills missing notebook packages.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--no-cache-dir",
    "datasets",
    "soundfile",
    "librosa",
    "numpy",
])


## 3. Declare Storage Paths

Colab stores heavy artifacts in Google Drive. Local Mac runs store them in ignored repo folders so the workflow remains reproducible without Colab.


In [ ]:
from pathlib import Path
import importlib.util
import os


def running_in_colab() -> bool:
    return importlib.util.find_spec("google.colab") is not None


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "shrike7").exists():
            return candidate
    raise RuntimeError("Could not find the Shrike-7 repo root from the current notebook path.")


IN_COLAB = running_in_colab() if EXECUTION_MODE == "auto" else EXECUTION_MODE == "colab"

if IN_COLAB:
    if USE_DRIVE_STORAGE:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        STORAGE_ROOT = Path("/content/drive/MyDrive/shrike-7")
    else:
        STORAGE_ROOT = Path("/content/shrike-7")
else:
    REPO_DIR = find_repo_root(Path.cwd())
    os.chdir(REPO_DIR)
    STORAGE_ROOT = REPO_DIR

if IN_COLAB:
    NOISE_ROOT = STORAGE_ROOT / "datasets" / "noise_for_boh"
else:
    NOISE_ROOT = STORAGE_ROOT / "data" / "noise_for_boh"

NOISE_DIR = NOISE_ROOT / "wav"
MANIFEST = NOISE_ROOT / "manifest.jsonl"
CONFIG_SNAPSHOT = NOISE_ROOT / "noise_collection_config.json"

NOISE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Execution mode: {'colab' if IN_COLAB else 'local'}")
print(f"Storage root: {STORAGE_ROOT}")
print(f"Noise wav dir: {NOISE_DIR}")
print(f"Manifest path: {MANIFEST}")


## 4. Helper functions

These helpers keep the generated files consistent: 16 kHz mono, float32 audio, JSONL manifest rows, and reproducible synthetic noise.

In [ ]:
from datetime import datetime, timezone
import json
import random

import librosa
import numpy as np
import soundfile as sf
from datasets import load_dataset

random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)


def write_wav_and_row(rows: list[dict], filename: str, audio: np.ndarray, source: str, label: str) -> None:
    audio = np.asarray(audio, dtype=np.float32)
    sf.write(NOISE_DIR / filename, audio, SAMPLE_RATE)
    rows.append(
        {
            "path": f"wav/{filename}",
            "source": source,
            "label": label,
            "duration_s": len(audio) / SAMPLE_RATE,
            "sampling_rate": SAMPLE_RATE,
            "expected_transcript": "",
        }
    )


def pink_noise(n_samples: int, amplitude: float, rng: np.random.Generator) -> np.ndarray:
    white = rng.normal(0.0, 1.0, n_samples)
    freqs = np.fft.rfftfreq(n_samples, d=1.0 / SAMPLE_RATE)
    spectrum = np.fft.rfft(white)
    scale = np.ones_like(freqs)
    scale[1:] = 1.0 / np.sqrt(freqs[1:])
    pink = np.fft.irfft(spectrum * scale, n=n_samples)
    pink = pink / max(np.max(np.abs(pink)), 1e-8)
    return (pink * amplitude).astype(np.float32)


def target_reached(rows: list[dict]) -> bool:
    return len(rows) >= TARGET_TOTAL_SAMPLES

## 5. Collect ESC-50 environmental sounds

Human-voice-like categories are excluded where possible. The source and label are preserved for auditability.

In [ ]:
rows: list[dict] = []

esc50_cfg = NOISE_SOURCES["esc50"]
if esc50_cfg["enabled"] and not target_reached(rows):
    count = 0
    exclude_categories = set(esc50_cfg["exclude_categories"])
    esc50 = load_dataset(esc50_cfg["hf_repo"], split=esc50_cfg["split"], streaming=True)

    for ex in esc50:
        category = ex.get("category", "")
        if category in exclude_categories:
            continue

        audio = ex["audio"]
        sr = audio["sampling_rate"]
        arr = audio["array"].astype(np.float32)
        if sr != SAMPLE_RATE:
            arr = librosa.resample(arr, orig_sr=sr, target_sr=SAMPLE_RATE)

        filename = f"esc50_{count:04d}.wav"
        write_wav_and_row(rows, filename, arr, "esc50", category)

        count += 1
        if count >= esc50_cfg["n_samples"] or target_reached(rows):
            break

print(f"Rows after ESC-50: {len(rows)}")

## 6. Generate synthetic non-speech audio

Synthetic silence, white noise, and pink noise create controlled edge cases for ASR hallucination probing.

In [ ]:
synthetic_cfg = SYNTHETIC_NOISE

if synthetic_cfg["enabled"] and not target_reached(rows):
    for kind, n_items in [
        ("silence", synthetic_cfg["n_silence"]),
        ("white_noise", synthetic_cfg["n_white_noise"]),
        ("pink_noise", synthetic_cfg["n_pink_noise"]),
    ]:
        for idx in range(n_items):
            if target_reached(rows):
                break

            duration = float(rng.choice(synthetic_cfg["durations_s"]))
            amplitude = float(rng.choice(synthetic_cfg["amplitudes"]))
            n_samples = int(duration * SAMPLE_RATE)

            if kind == "silence":
                arr = np.zeros(n_samples, dtype=np.float32)
            elif kind == "white_noise":
                arr = rng.normal(0.0, amplitude, n_samples).astype(np.float32)
            elif kind == "pink_noise":
                arr = pink_noise(n_samples, amplitude, rng)
            else:
                raise ValueError(f"Unsupported synthetic noise kind: {kind}")

            filename = f"synthetic_{kind}_{idx:04d}.wav"
            write_wav_and_row(rows, filename, arr, f"synthetic_{kind}", kind)

print(f"Rows after synthetic generation: {len(rows)}")

## 7. Save manifest and config snapshot

Notebook 02 consumes `manifest.jsonl`. The config snapshot records exactly how this dataset was produced.

In [ ]:
with MANIFEST.open("w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

snapshot = {
    "project": PROJECT,
    "run_name": RUN_NAME,
    "seed": SEED,
    "sample_rate": SAMPLE_RATE,
    "target_total_samples": TARGET_TOTAL_SAMPLES,
    "num_rows": len(rows),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "noise_sources": NOISE_SOURCES,
    "synthetic_noise": SYNTHETIC_NOISE,
    "manifest": str(MANIFEST),
    "noise_dir": str(NOISE_DIR),
}
CONFIG_SNAPSHOT.write_text(json.dumps(snapshot, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Saved {len(rows)} non-speech samples to {NOISE_DIR}")
print(f"Manifest: {MANIFEST}")
print(f"Config snapshot: {CONFIG_SNAPSHOT}")